First of all, we create and configure the Spark session using Apache Spark.

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [16]:
spark = SparkSession.builder \
    .appName("CleaningTransactions") \
    .master("local[*]") \
    .getOrCreate()

In [17]:
spark.sparkContext.setLogLevel("ERROR")

Once the dataset has been loaded, we proceed with its cleaning and preparation for the ETL process.

In [26]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("sep", ";") \
    .csv("../datasets/dataset_transactions.csv", header=True, inferSchema=True)

First of all, we select the columns of interest and we rename them.

In [29]:
df = df.drop("CODCOMUNIDAD", "CODPROVINCIA")

In [31]:
df = df.withColumnRenamed("COMUNIDAD", "Region") \
    .withColumnRenamed("PROVINCIA", "City") \
    .withColumnRenamed("Año", "Year") \
    .withColumnRenamed("Numero_Transacciones", "Total_Transactions") \
    .withColumnRenamed("Valor_Transacciones", "Total_Price") \
    .withColumnRenamed("Trimestre", "Quarter")

In [37]:
df.show(truncate=False)

+---------+---------+------------+----+-------+------------------+-----------+
|Region   |City     |Tipo        |Year|Quarter|Total_Transactions|Total_Price|
+---------+---------+------------+----+-------+------------------+-----------+
|Cantabria|Cantabria|Segunda Mano|2004|1      |1642              |183685000  |
|Cantabria|Cantabria|Segunda Mano|2005|1      |2104              |282850500  |
|Cantabria|Cantabria|Segunda Mano|2009|1      |499               |82172800   |
|Cantabria|Cantabria|Segunda Mano|2007|1      |1694              |292133900  |
|Cantabria|Cantabria|Segunda Mano|2006|1      |1895              |297149200  |
|Cantabria|Cantabria|Segunda Mano|2012|1      |482               |69979500   |
|Cantabria|Cantabria|Segunda Mano|2024|1      |2105              |321070700  |
|Cantabria|Cantabria|Segunda Mano|2014|1      |659               |87944000   |
|Cantabria|Cantabria|Segunda Mano|2008|1      |852               |159169000  |
|Cantabria|Cantabria|Segunda Mano|2010|1      |703  

Now we can see that some columns are not in the format we need, so we will transform them.

In [39]:
df.select("Region").distinct().orderBy("Region").show(truncate=False)

+---------------------------+
|Region                     |
+---------------------------+
|Andalucía                  |
|Aragón                     |
|Asturias, Principado de    |
|Balears, Illes             |
|Canarias                   |
|Cantabria                  |
|Castilla y León            |
|Castilla-La Mancha         |
|Cataluña                   |
|Ceuta                      |
|Comunitat Valenciana       |
|Extremadura                |
|Galicia                    |
|Madrid, Comunidad de       |
|Melilla                    |
|Murcia, Región de          |
|Navarra, Comunidad Foral de|
|País Vasco                 |
|Rioja, La                  |
+---------------------------+



In [40]:
df = df.withColumn("Region",
    F.when(
        F.col("Region").contains(","),
        F.concat_ws(
            " ",
            F.trim(F.element_at(F.split(F.col("Region"), ","), 2)),
            F.trim(F.element_at(F.split(F.col("Region"), ","), 1))
        )
    ).otherwise(F.col("Region"))
)

In [41]:
df.select("Region").distinct().orderBy("Region").show(truncate=False)

+--------------------------+
|Region                    |
+--------------------------+
|Andalucía                 |
|Aragón                    |
|Canarias                  |
|Cantabria                 |
|Castilla y León           |
|Castilla-La Mancha        |
|Cataluña                  |
|Ceuta                     |
|Comunidad Foral de Navarra|
|Comunidad de Madrid       |
|Comunitat Valenciana      |
|Extremadura               |
|Galicia                   |
|Illes Balears             |
|La Rioja                  |
|Melilla                   |
|País Vasco                |
|Principado de Asturias    |
|Región de Murcia          |
+--------------------------+



In [42]:
df = df.withColumn(
    "Region",
    F.when(F.col("Region") == "Castilla-La Mancha", "Castilla - La Mancha")
     .otherwise(F.col("Region"))
)

In [43]:
df.select("Region").distinct().orderBy("Region").show(truncate=False)

+--------------------------+
|Region                    |
+--------------------------+
|Andalucía                 |
|Aragón                    |
|Canarias                  |
|Cantabria                 |
|Castilla - La Mancha      |
|Castilla y León           |
|Cataluña                  |
|Ceuta                     |
|Comunidad Foral de Navarra|
|Comunidad de Madrid       |
|Comunitat Valenciana      |
|Extremadura               |
|Galicia                   |
|Illes Balears             |
|La Rioja                  |
|Melilla                   |
|País Vasco                |
|Principado de Asturias    |
|Región de Murcia          |
+--------------------------+



In [44]:
df = df.withColumn("City",
    F.when(
        F.col("City").contains(","),
        F.concat_ws(
            " ",
            F.trim(F.element_at(F.split(F.col("City"), ","), 2)),
            F.trim(F.element_at(F.split(F.col("City"), ","), 1))
        )
    ).otherwise(F.col("City"))
)

In [45]:
df.select("City").distinct().orderBy("City").show(truncate=False)

+------------------+
|City              |
+------------------+
|A Coruña          |
|Albacete          |
|Alicante/Alacant  |
|Almería           |
|Araba/Álava       |
|Asturias          |
|Badajoz           |
|Barcelona         |
|Bizkaia           |
|Burgos            |
|Cantabria         |
|Castellón/Castelló|
|Ceuta             |
|Ciudad Real       |
|Cuenca            |
|Cáceres           |
|Cádiz             |
|Córdoba           |
|Gipuzkoa          |
|Girona            |
+------------------+
only showing top 20 rows


In [47]:
df.orderBy("Region", "City", "Year", "Quarter").show(truncate=False)

+---------+-------+----------------------+----+-------+------------------+-----------+
|Region   |City   |Tipo                  |Year|Quarter|Total_Transactions|Total_Price|
+---------+-------+----------------------+----+-------+------------------+-----------+
|Andalucía|Almería|Segunda Mano          |2004|1      |2281              |177703300  |
|Andalucía|Almería|Nueva                 |2004|1      |2110              |199291600  |
|Andalucía|Almería|Protegida Nueva       |2004|1      |38                |NULL       |
|Andalucía|Almería|Protegida Segunda Mano|2004|1      |73                |NULL       |
|Andalucía|Almería|Segunda Mano          |2004|2      |3058              |239721500  |
|Andalucía|Almería|Nueva                 |2004|2      |2487              |227464800  |
|Andalucía|Almería|Protegida Nueva       |2004|2      |106               |NULL       |
|Andalucía|Almería|Protegida Segunda Mano|2004|2      |80                |NULL       |
|Andalucía|Almería|Segunda Mano          |2

In [46]:
df.select("Tipo").distinct().show(truncate=False)

+----------------------+
|Tipo                  |
+----------------------+
|Protegida Nueva       |
|Segunda Mano          |
|Protegida Segunda Mano|
|Nueva                 |
+----------------------+



In [48]:
df = df.withColumn(
    "Regime",
    F.when(F.col("Tipo").isin("Protegida Nueva", "Protegida Segunda Mano"), "Protegida")
    .otherwise("Libre")
)

In [49]:
df = df.withColumn(
    "Condition",
    F.when(F.col("Tipo").isin("Protegida Nueva", "Nueva"), "Nueva")
     .otherwise("Segunda mano")
)

In [51]:
df = df.drop("Tipo")

In [52]:
df.show(truncate=False)

+---------+---------+----+-------+------------------+-----------+------+------------+
|Region   |City     |Year|Quarter|Total_Transactions|Total_Price|Regime|Condition   |
+---------+---------+----+-------+------------------+-----------+------+------------+
|Cantabria|Cantabria|2004|1      |1642              |183685000  |Libre |Segunda mano|
|Cantabria|Cantabria|2005|1      |2104              |282850500  |Libre |Segunda mano|
|Cantabria|Cantabria|2009|1      |499               |82172800   |Libre |Segunda mano|
|Cantabria|Cantabria|2007|1      |1694              |292133900  |Libre |Segunda mano|
|Cantabria|Cantabria|2006|1      |1895              |297149200  |Libre |Segunda mano|
|Cantabria|Cantabria|2012|1      |482               |69979500   |Libre |Segunda mano|
|Cantabria|Cantabria|2024|1      |2105              |321070700  |Libre |Segunda mano|
|Cantabria|Cantabria|2014|1      |659               |87944000   |Libre |Segunda mano|
|Cantabria|Cantabria|2008|1      |852               |1

In [ ]:
df.coalesce(1).write \
    .mode("append") \
    .option("header", True) \
    .csv("../datasets_def/transactions_clean")